# Patches — Bloc 2, Sessió 2
### El so com a fitxer: WAV, mixing i loops

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Colab el carregarà directament des de GitHub. Si vols guardar els teus canvis: `File -> Save a copy in Drive`.

Aquest notebook és per a la **demo guiada en directe**. Executa cada cel·la i escolta el resultat abans de passar a la següent.

In [ ]:
import numpy as np
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio

sample_rate = 44100

## 0. Carregar els fitxers d'aquesta sessió

Els fitxers d'àudio són al repo, a `recursos/audio/`. Com que Colab carrega el notebook des de GitHub, per accedir als fitxers els descarreguem amb una URL `raw`.

In [ ]:
import urllib.request

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/02_numpy_audio/sessio02_wav_io/recursos/audio/"

for fname in ["perc_loop.wav", "example_pad.wav"]:
    urllib.request.urlretrieve(base_url + fname, fname)

print("Fitxers descarregats:", "perc_loop.wav", "example_pad.wav")

## 1. Recordatori Sessió 1 — generar un to

Reaprofitem la funció `generate_tone` de la setmana passada.

In [ ]:
def generate_tone(freq, duration, amplitude=0.5, sample_rate=44100):
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    wave = amplitude * np.sin(2 * np.pi * freq * t)
    return wave

tone = generate_tone(freq=440, duration=1.0, amplitude=0.5)
Audio(tone, rate=sample_rate)

## 2. Escriure un WAV

Desem aquest array a disc.

In [ ]:
sf.write("el_meu_to.wav", tone, sample_rate)
print("Fitxer escrit!")

## 3. Llegir-lo de nou

🎤 *Pregunta a la classe: si comparem `tone` (l'original) amb `data` (el llegit), seran exactament iguals?*

In [ ]:
data, sr = sf.read("el_meu_to.wav")

print("Sample rate:", sr)
print("Longitud original:", len(tone), "  Longitud llegida:", len(data))
print("Són (gairebé) iguals?", np.allclose(tone, data, atol=1e-4))

Audio(data, rate=sr)

## 4. Carregar i visualitzar el loop de percussió

In [ ]:
perc, sr_perc = librosa.load("perc_loop.wav", sr=None)

print("Durada:", len(perc) / sr_perc, "segons")

librosa.display.waveshow(perc, sr=sr_perc)
plt.title("Loop de percussió")
plt.show()

Audio(perc, rate=sr_perc)

## 5. DEMO EN DIRECTE — Mixing

🎤 *Pregunta: `perc` té 88200 mostres (2s). El nostre `tone` en té 44100 (1s). Què passarà si fem `perc + tone` directament?*

(Resposta: error — longituds diferents. Cal igualar-les primer.)

In [ ]:
print("len(perc):", len(perc))
print("len(tone):", len(tone))

# Igualem longituds retallant pel mes curt
n = min(len(perc), len(tone))
mix = perc[:n] + tone[:n]

# Normalitzem per evitar clipping
mix = mix / np.max(np.abs(mix))

Audio(mix, rate=sample_rate)

## 6. DEMO EN DIRECTE — Loop amb np.tile

🎤 *Pregunta: si `perc` dura 2 segons, quant durarà `np.tile(perc, 4)`?*

In [ ]:
perc_x4 = np.tile(perc, 4)

print("Durada x4:", len(perc_x4) / sr_perc, "segons")

Audio(perc_x4, rate=sr_perc)

## 7. Tot junt: loop + to per sobre, amb fade

Combinem el loop repetit amb el `pad` d'exemple (un altre fitxer proporcionat), aplicant gain i fade — tot reaprofitant funcions de la Sessió 1.

In [ ]:
pad, sr_pad = librosa.load("example_pad.wav", sr=None)

# Repetim el loop de percussio per cobrir la durada del pad
n_repeats = int(np.ceil(len(pad) / len(perc)))
perc_repeated = np.tile(perc, n_repeats)[:len(pad)]

# Gain: baixem un poc la percussio
perc_repeated = perc_repeated * 0.6

# Mixing
mix_final = perc_repeated + pad
mix_final = mix_final / np.max(np.abs(mix_final))

librosa.display.waveshow(mix_final, sr=sr_pad)
plt.title("Mix final")
plt.show()

Audio(mix_final, rate=sr_pad)